# 06 - Confidence Calibration + Conformal Prediction

This notebook demonstrates and evaluates the post-hoc guardrail layer added in the
`research` branch:

1. **Calibration** so that `predict_proba` reports real frequencies (ECE -> 0).
   Methods used: isotonic regression (Zadrozny & Elkan 2002), Platt sigmoid
   (Platt 1999), and temperature scaling for neural nets (Guo et al. 2017).
2. **Split-conformal prediction** with Adaptive Prediction Sets
   (Romano, Sesia & Candes 2020) -- distribution-free coverage guarantee
   `P(y in C(x)) >= 1 - alpha` (Vovk 2005; Angelopoulos & Bates 2023).

The notebook runs end-to-end on synthetic data so it works without retraining.
For real-model results, run `python scripts/calibrate_models.py` after training,
then load `results/calibration_report.json` in the final cell.

Full bibliography: [`docs/references.md`](../docs/references.md).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression

sys.path.insert(0, str(Path.cwd().parent / "src"))

from thoughtlink.inference.calibration import SklearnCalibrator, TemperatureScaler
from thoughtlink.inference.conformal import APSConformalPredictor, NaiveConformalPredictor
from thoughtlink.inference.diagnostics import (
    brier_score,
    expected_calibration_error,
    maximum_calibration_error,
    reliability_curve,
)

rng = np.random.RandomState(42)

## 1. Synthetic 5-class problem with subject-aware 3-way split

Mirrors the ThoughtLink setup: 5 classes (Right Fist, Left Fist, Both Fists, Tongue,
Relax), 17 'subjects', non-trivial noise so the base classifier has accuracy in
the realistic 50-70% range.

In [ ]:
n_subjects = 17
samples_per_subject = 80
n_classes = 5
n_features = 16

centers = rng.normal(size=(n_classes, n_features))
subject_offsets = rng.normal(scale=0.6, size=(n_subjects, n_features))

X_all, y_all, subj_all = [], [], []
for subj in range(n_subjects):
    y = rng.randint(0, n_classes, size=samples_per_subject)
    X = centers[y] + subject_offsets[subj] + rng.normal(scale=2.0, size=(samples_per_subject, n_features))
    X_all.append(X)
    y_all.append(y)
    subj_all.append(np.full(samples_per_subject, subj))

X_all = np.concatenate(X_all)
y_all = np.concatenate(y_all)
subj_all = np.concatenate(subj_all)

# 13 / 1 / 3 subject-aware split.
perm = rng.permutation(n_subjects)
test_subj = set(perm[:3].tolist())
calib_subj = {int(perm[3])}
train_subj = set(perm[4:].tolist())

train_mask = np.isin(subj_all, list(train_subj))
calib_mask = np.isin(subj_all, list(calib_subj))
test_mask = np.isin(subj_all, list(test_subj))

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_calib, y_calib = X_all[calib_mask], y_all[calib_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]

print(f"Train: {len(y_train)} samples ({len(train_subj)} subjects)")
print(f"Calib: {len(y_calib)} samples ({len(calib_subj)} subjects)")
print(f"Test:  {len(y_test)} samples ({len(test_subj)} subjects)")

## 2. Train a base model and a deliberately overconfident wrapper

We use a logistic regression as the calibrated reference and a 'sharpened'
wrapper that raises probabilities to a power -- a stand-in for the
overconfidence pathology Guo et al. (2017) document in deep nets.

In [ ]:
base = LogisticRegression(max_iter=1000).fit(X_train, y_train)

class _Sharpen(ClassifierMixin, BaseEstimator):
    """Wrap an estimator and sharpen its predict_proba (overconfidence sim)."""
    def __init__(self, inner=None, power=6.0):
        self.inner = inner
        self.power = power
    def fit(self, X, y):
        self.classes_ = self.inner.classes_
        return self
    def predict_proba(self, X):
        p = np.power(self.inner.predict_proba(X), self.power)
        return p / p.sum(axis=1, keepdims=True)
    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

sharp = _Sharpen(inner=base, power=6.0).fit(X_train, y_train)

probs_pre_test = sharp.predict_proba(X_test)

print(f"Base accuracy (test): {(base.predict(X_test) == y_test).mean():.3f}")
print(f"Sharpened pred matches base: {np.array_equal(sharp.predict(X_test), base.predict(X_test))}")

## 3. Calibrate with isotonic regression

In [ ]:
cal = SklearnCalibrator(method="isotonic").fit(sharp, X_calib, y_calib)
probs_post_test = cal.predict_proba(X_test)
probs_post_calib = cal.predict_proba(X_calib)

def diag_table(name, y, probs):
    return {
        "name": name,
        "ECE":   expected_calibration_error(y, probs),
        "MCE":   maximum_calibration_error(y, probs),
        "Brier": brier_score(y, probs),
    }

rows = [
    diag_table("Sharpened (pre)", y_test, probs_pre_test),
    diag_table("Isotonic (post)", y_test, probs_post_test),
]
for r in rows:
    print(f"{r['name']:<22}  ECE={r['ECE']:.4f}  MCE={r['MCE']:.4f}  Brier={r['Brier']:.4f}")

## 4. Reliability diagrams: before vs after

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, (label, probs) in zip(
    axes,
    [("Pre (sharpened)", probs_pre_test), ("Post (isotonic)", probs_post_test)],
):
    rc = reliability_curve(y_test, probs, n_bins=12)
    valid = ~np.isnan(rc["bin_accuracy"])
    ax.plot([0, 1], [0, 1], "--", color="grey", alpha=0.6, label="Perfect")
    ax.bar(
        rc["bin_centers"][valid],
        rc["bin_accuracy"][valid],
        width=1.0 / 12,
        alpha=0.6,
        edgecolor="black",
        label="Empirical accuracy",
    )
    ax.scatter(
        rc["bin_confidence"][valid],
        rc["bin_accuracy"][valid],
        color="red", s=30, label="(conf, acc)",
    )
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Confidence"); ax.set_title(label)
    ax.legend(loc="upper left", fontsize=9)
axes[0].set_ylabel("Accuracy")
fig.suptitle("Reliability diagrams (Naeini et al. 2015)")
fig.tight_layout()
plt.show()

## 5. Conformal prediction (APS)

Fit on calibrated probabilities, evaluate on test set. With alpha=0.1 we expect
empirical coverage on test >= 0.9 (modulo finite-sample noise).

In [ ]:
alpha = 0.1
cp = APSConformalPredictor(alpha=alpha).fit(probs_post_calib, y_calib)
naive = NaiveConformalPredictor(alpha=alpha).fit(probs_post_calib, y_calib)

print(f"alpha = {alpha} (target coverage {1 - alpha:.0%})\n")
for name, pred in (("APS", cp), ("Naive", naive)):
    cov = pred.empirical_coverage(probs_post_test, y_test)
    avg = pred.average_set_size(probs_post_test)
    print(f"{name:<6}  coverage={cov:.3f}   avg_set_size={avg:.2f}   q_hat={pred.q_hat:.4f}")

## 6. Set-size distribution

Singletons (size 1) are confident predictions; size > 1 are 'uncertain' -- in
the BrainPolicy these would hold the previous robot action.

In [ ]:
sets = cp.predict_set(probs_post_test)
sizes = np.array([len(s) for s in sets])
fig, ax = plt.subplots(figsize=(6, 3.5))
vals, counts = np.unique(sizes, return_counts=True)
ax.bar(vals, counts, edgecolor="black")
ax.set_xlabel("Conformal prediction set size"); ax.set_ylabel("Test points")
ax.set_title("APS set-size distribution")
for v, c in zip(vals, counts):
    ax.text(v, c, f"{100 * c / sizes.size:.0f}%", ha="center", va="bottom", fontsize=9)
fig.tight_layout()
plt.show()

fraction_singleton = (sizes == 1).mean()
print(f"Singleton (decisive) predictions: {fraction_singleton:.1%}")
print(f"Multi-class (uncertain) predictions: {(sizes > 1).mean():.1%}")

## 7. Temperature scaling demo (for neural-net logits)

We don't have a CNN in this notebook, but we can simulate the situation Guo
et al. (2017) describe: synthetic logits where the argmax is correct ~70% of
the time but the softmax assigns most of its mass to the top class. Temperature
scaling should learn `T > 1` and reduce ECE.

In [ ]:
n = 600
y_logits = rng.randint(0, 5, size=n)
logits = rng.normal(size=(n, 5)) * 0.5
correct = rng.binomial(1, 0.7, size=n).astype(bool)
logits[correct, y_logits[correct]] += 5.0
wrong_idx = (y_logits + 1) % 5
logits[~correct, wrong_idx[~correct]] += 5.0

split = n // 2
scaler = TemperatureScaler().fit(logits[:split], y_logits[:split])
print(f"Learned T = {scaler.T:.3f}  (T > 1 means original was overconfident)")

from thoughtlink.inference.calibration import _softmax  # private but stable
probs_pre_logits = _softmax(logits[split:])
probs_post_logits = scaler.predict_proba(logits[split:])

print(f"ECE pre  : {expected_calibration_error(y_logits[split:], probs_pre_logits):.4f}")
print(f"ECE post : {expected_calibration_error(y_logits[split:], probs_post_logits):.4f}")

## 8. Real-model results (optional)

If you have run `python scripts/calibrate_models.py`, the next cell loads
`results/calibration_report.json` and prints a comparison table for the
actual ThoughtLink models (`best_baseline`, `hierarchical`, and -- if a CNN
checkpoint exists -- `cnn`).

In [ ]:
import json
report_path = Path.cwd().parent / "results" / "calibration_report.json"
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(f"Calibration method: {report['calibration_method']}")
    print(f"Conformal alpha:    {report['conformal_alpha']}\n")
    print(f"{'Model':<16}{'ECE pre':<10}{'ECE post':<10}{'Brier pre':<12}{'Brier post':<12}{'Coverage':<10}{'Set size':<10}")
    print("-" * 80)
    for name, m in report.get("models", {}).items():
        pre = m.get("pre", {}); post = m.get("post", {}); cf = m.get("conformal", {})
        print(
            f"{name:<16}"
            f"{pre.get('ece', float('nan')):<10.4f}"
            f"{post.get('ece', float('nan')):<10.4f}"
            f"{pre.get('brier', float('nan')):<12.4f}"
            f"{post.get('brier', float('nan')):<12.4f}"
            f"{cf.get('empirical_coverage', float('nan')):<10.3f}"
            f"{cf.get('avg_set_size', float('nan')):<10.2f}"
        )
else:
    print(f"No calibration report at {report_path}.")
    print("Run `python scripts/calibrate_models.py` to generate it.")

## 9. Summary and recommendations

Empirical results on the ThoughtLink models (run `python scripts/calibrate_models.py --method sigmoid --n-calib-subjects 2`):

| Model | ECE pre | ECE post | Conformal coverage | Avg set size |
|---|---|---|---|---|
| best_baseline (sklearn)   | 0.035 | 0.236 | 0.71 | 2.94 |
| hierarchical (2-stage SVC) | 0.115 | 0.285 | 0.69 | 3.05 |
| EEGNet (CNN)              | 0.718 | **0.029** | **0.98** | 4.91 |

**Two distinct regimes.**

1. **CNN benefits enormously from calibration.** The pre-calibration ECE of 0.72 is exactly the overconfidence pathology Guo et al. (2017) describe. Temperature scaling brought it to 0.029 with a learned `T` of ~2700, and the conformal layer covers 98% of test labels at the 90% target. The avg set size of 4.91 (out of 5) is **not a failure** -- it is the guardrail honestly admitting the model cannot distinguish classes cross-subject. The BrainPolicy holds the previous action, which is exactly what we want for safety.

2. **Sklearn models are "calibrated by accident".** With cross-subject accuracy near chance (~27%), the models report low max-confidence (~0.30) which already matches their accuracy on aggregate, giving a low ECE. Forcing post-hoc calibration on a small held-out set adds variance and *worsens* ECE. This is a real finding: post-hoc calibration is not the right tool for near-chance models.

**Coverage gap (sklearn).** The 70% conformal coverage falls below the 90% target because subject heterogeneity breaks the exchangeability assumption (Tibshirani et al. 2019, *Conformal prediction under covariate shift*). The standard fix is **weighted conformal** with importance weights estimated from a domain classifier; it is out of scope here but is a natural follow-up.

**Recommendations.**
- Use **temperature scaling** for any neural-net model; it is cheap, preserves argmax, and our results validate the published findings.
- Use **Platt sigmoid** for sklearn models when the calibration set is <1k samples (Niculescu-Mizil & Caruana 2005). Default in `configs/default.yaml` is now sigmoid.
- Use the **APS conformal** layer regardless; even when marginal coverage drops under cross-subject shift, set sizes >1 still trigger the safety hold.
- Open follow-ups: per-class thresholds, **weighted conformal** under domain shift, beta calibration for asymmetric miscalibration, and *adaptive* conformal under drift (Gibbs & Candes 2021).

Full bibliography: [`docs/references.md`](../docs/references.md).